# Approach 1

In [3]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error
import matplotlib.pyplot as plt
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# =====================================================================
# LOAD DATA
# =====================================================================
df = pd.read_csv(r'../data/raw/ecommerce_price_prediction-train.csv')
df['capturedAt'] = pd.to_datetime(df['capturedAt'])
df['scrape_date'] = df['capturedAt'].dt.date

outage_date = df['scrape_date'].max()

train_df = df[df['scrape_date'] < outage_date].copy()
val_df   = df[df['scrape_date'] == outage_date].copy()

print(f"Train : {train_df.shape[0]:,} rows")
print(f"Val   : {val_df.shape[0]:,} rows  (outage day: {outage_date})")


# =====================================================================
# FREQUENCY MAP — dihitung dari TRAIN, bukan val (anti-leakage)
# =====================================================================
shop_freq_map = train_df['shopId'].value_counts()

# =====================================================================
# PREPARE FEATURES
# =====================================================================
def prepare_features(data: pd.DataFrame, shop_freq_map: pd.Series, is_train: bool = True) -> pd.DataFrame:
    df_feat = data.copy()

    # 1. Temporal cyclical
    df_feat['hour']     = df_feat['capturedAt'].dt.hour
    df_feat['hour_sin'] = np.sin(2 * np.pi * df_feat['hour'] / 24.0)
    df_feat['hour_cos'] = np.cos(2 * np.pi * df_feat['hour'] / 24.0)

    # 2. Pricing features
    denom = df_feat['item_price_max'] - df_feat['item_price_min']
    df_feat['price_position'] = np.where(
        denom == 0, 0.5,
        (df_feat['priceBeforeDiscount'] - df_feat['item_price_min']) / (denom + 1e-5)
    )
    df_feat['discount_ratio'] = np.where(
        df_feat['priceBeforeDiscount'] == 0, 0.0,
        df_feat['raw_discount'] / (df_feat['priceBeforeDiscount'] + 1e-5)
    )

    # 3. Frequency encoding — selalu dari train map (fix leakage)
    df_feat['shop_appearance_count'] = df_feat['shopId'].map(shop_freq_map).fillna(0)

    # 4. Kategorical
    df_feat['brand'] = df_feat['brand'].fillna('Unknown')
    cat_cols = [
        'shopId', 'itemId', 'modelId', 'cat_id', 'brand',
        'is_free_shipping', 'is_pre_order', 'is_official_shop',
        'is_verified', 'is_preferred_plus_seller'
    ]
    for col in cat_cols:
        if col in df_feat.columns:
            df_feat[col] = df_feat[col].astype('category')

    # 5. Drop kolom tidak relevan
    drop_cols = ['capturedAt', 'hour', 'scrape_date', 'stock', 'normal_stock', 'promotionId']
    df_feat = df_feat.drop(columns=[c for c in drop_cols if c in df_feat.columns])

    return df_feat

# =====================================================================
# SIMULASI OUTAGE — SESUAI KONDISI TEST YANG NYATA
# =====================================================================
print("\n[INFO] Simulating outage condition on validation day...")

OUTAGE_COLS = ['capturedAt', 'shopId', 'itemId', 'modelId']

# val_df di training CSV semuanya punya price
# Tapi kondisi test: hanya 100 baris per hari yang punya price (anchor)
# Jadi kita PAKSA simulasi persis seperti test:
# - Ambil 100 sebagai anchor (punya price)  
# - Sisanya: buang semua kolom kecuali OUTAGE_COLS (hapus price juga)

anchor_set_raw  = val_df.sample(n=100, random_state=RANDOM_SEED).copy()
hidden_test_raw = val_df.drop(index=anchor_set_raw.index).copy()

# Simpan ground truth SEBELUM di-strip (untuk evaluasi — hanya ada di validasi)
y_hidden_true = hidden_test_raw['price'].copy()
y_anchor_true = anchor_set_raw['price'].values

# Simulasi kondisi test: hidden test hanya punya OUTAGE_COLS (price di-drop)
hidden_test_outage = hidden_test_raw[OUTAGE_COLS].copy()  # price tidak ada!

# =====================================================================
# BACKFILL — LOCF dari train per itemId
# =====================================================================
missing_raw_cols = [
    col for col in train_df.columns
    if col not in OUTAGE_COLS + ['price', 'scrape_date']
]

last_known_raw = (
    train_df.groupby('itemId')[missing_raw_cols]
    .last()
    .reset_index()
    .drop_duplicates(subset='itemId')
)

hidden_test_backfilled = hidden_test_outage.merge(last_known_raw, on='itemId', how='left')

# Safety net cold-start
cold_start_count = 0
for col in missing_raw_cols:
    n_missing = hidden_test_backfilled[col].isna().sum()
    if n_missing > 0:
        cold_start_count += n_missing
        if pd.api.types.is_numeric_dtype(train_df[col]):
            hidden_test_backfilled[col] = hidden_test_backfilled[col].fillna(train_df[col].median())
        else:
            hidden_test_backfilled[col] = hidden_test_backfilled[col].fillna(train_df[col].mode()[0])

print(f"  Cold-start cells patched : {cold_start_count:,}")
print(f"  Anchor set size                        : {len(anchor_set_raw)}")
print(f"  Hidden test size                       : {len(hidden_test_backfilled)}")


# =====================================================================
# FEATURE ENGINEERING
# =====================================================================
train_processed          = prepare_features(train_df,              shop_freq_map, is_train=True)
anchor_processed         = prepare_features(anchor_set_raw,        shop_freq_map, is_train=False)
hidden_test_processed    = prepare_features(hidden_test_backfilled, shop_freq_map, is_train=False)

FEATURES = [col for col in train_processed.columns if col not in ['price', 'log_price']]

X_train        = train_processed[FEATURES]
y_train_log    = np.log1p(train_processed['price'])

X_anchor       = anchor_processed[FEATURES]
y_anchor_true  = anchor_set_raw['price'].values

X_hidden       = hidden_test_processed[FEATURES]

# =====================================================================
# OPTUNA TUNING
# =====================================================================
def objective(trial):
    params = {
        'n_estimators'     : 300,
        'learning_rate'    : trial.suggest_float('learning_rate', 0.01, 0.1, step=0.01),
        'num_leaves'       : trial.suggest_int('num_leaves', 31, 127),
        'max_depth'        : trial.suggest_int('max_depth', 6, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'random_state'     : RANDOM_SEED,
        'n_jobs'           : -1,
        'verbose'          : -1,
    }
    model = lgb.LGBMRegressor(**params)
    model.fit(X_train, y_train_log)

    # Calibration dari anchor organik murni
    anchor_preds   = np.expm1(model.predict(X_anchor))
    calib_factor   = np.median(y_anchor_true / (anchor_preds + 1e-5))

    # Evaluasi di hidden test
    preds_base     = np.expm1(model.predict(X_hidden))
    preds_calib    = preds_base * calib_factor

    return mean_absolute_percentage_error(y_hidden_true, preds_calib)

print("\n[INFO] Starting Optuna tuning...")
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=20)

print(f"\nBest MAPE (outage-simulated, calibrated): {study.best_value*100:.2f}%")
print(f"Best params: {study.best_params}")


# =====================================================================
# EVALUASI FINAL
# =====================================================================
best_model = lgb.LGBMRegressor(
    n_estimators=300, random_state=RANDOM_SEED, n_jobs=-1, verbose=-1,
    **study.best_params
)
best_model.fit(X_train, y_train_log)

final_anchor_preds = np.expm1(best_model.predict(X_anchor))
final_calib_factor = np.median(y_anchor_true / (final_anchor_preds + 1e-5))

final_preds_base  = np.expm1(best_model.predict(X_hidden))
final_preds_calib = final_preds_base * final_calib_factor

print("\n--- FINAL EVALUATION (Outage Simulation) ---")
print(f"{'Metric':<6}  {'Baseline':>14}  {'Calibrated':>14}")
print("-" * 40)
for name, func in [
    ('MAE',  mean_absolute_error),
    ('RMSE', lambda y, p: np.sqrt(mean_squared_error(y, p))),
    ('MAPE', lambda y, p: mean_absolute_percentage_error(y, p) * 100),
]:
    base  = func(y_hidden_true, final_preds_base)
    calib = func(y_hidden_true, final_preds_calib)
    unit  = '%' if name == 'MAPE' else ''
    print(f"{name:<6}  {base:>13.2f}{unit}  {calib:>13.2f}{unit}")

improvement = (
    (mean_absolute_percentage_error(y_hidden_true, final_preds_base) -
     mean_absolute_percentage_error(y_hidden_true, final_preds_calib))
    / mean_absolute_percentage_error(y_hidden_true, final_preds_base) * 100
)
print(f"\nCalibration reduced MAPE by: {improvement:.2f}%")
print(f"Final calibration factor   : {final_calib_factor:.4f}")

Train : 301,355 rows
Val   : 4,871 rows  (outage day: 2025-03-22)

[INFO] Simulating outage condition on validation day...
  Cold-start cells patched : 12,315
  Anchor set size                        : 100
  Hidden test size                       : 4771

[INFO] Starting Optuna tuning...

Best MAPE (outage-simulated, calibrated): 1.21%
Best params: {'learning_rate': 0.08, 'num_leaves': 80, 'max_depth': 11, 'min_child_samples': 58}

--- FINAL EVALUATION (Outage Simulation) ---
Metric        Baseline      Calibrated
----------------------------------------
MAE         415674.82      415530.29
RMSE       2338620.36     2338473.11
MAPE             1.21%           1.21%

Calibration reduced MAPE by: -0.02%
Final calibration factor   : 1.0000


# Approach 2

In [4]:
# =====================================================================
# APPROACH 2 — ENTITY-CONDITIONED MODEL
# =====================================================================

# Hitung entity stats hanya dari train
shop_stats = train_df.groupby('shopId')['price'].agg(['mean', 'std']).reset_index()
shop_stats.columns = ['shopId', 'historical_shop_price_mean', 'historical_shop_price_std']

item_stats = train_df.groupby('itemId')['price'].agg(['mean']).reset_index()
item_stats.columns = ['itemId', 'historical_item_price_mean']

global_mean = train_df['price'].mean()

def add_entity_stats(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out = out.merge(shop_stats, on='shopId', how='left')
    out = out.merge(item_stats, on='itemId', how='left')
    out['historical_shop_price_mean'] = out['historical_shop_price_mean'].fillna(global_mean)
    out['historical_shop_price_std']  = out['historical_shop_price_std'].fillna(0)
    out['historical_item_price_mean'] = out['historical_item_price_mean'].fillna(global_mean)
    return out

# Pakai anchor_set_raw dan hidden_test_backfilled dari pipeline Approach 1
# (sudah konsisten, anchor organik, hidden test sudah di-backfill + strip)
train_tier2       = add_entity_stats(train_processed)
anchor_tier2      = add_entity_stats(anchor_processed)       # dari anchor_set_raw yg sudah diproses
hidden_test_tier2 = add_entity_stats(hidden_test_processed)  # dari hidden_test_backfilled yg sudah diproses

# Pasang kembali price untuk keperluan kalibrasi & evaluasi
anchor_tier2['price']      = y_anchor_true
hidden_test_tier2['price'] = y_hidden_true.values  # ground truth, hanya ada di validasi

FEATURES_T2 = [col for col in train_tier2.columns if col not in ['price', 'log_price']]

X_train_t2       = train_tier2[FEATURES_T2]
y_train_t2       = np.log1p(train_tier2['price'])
X_anchor_t2      = anchor_tier2[FEATURES_T2]
y_anchor_t2      = y_anchor_true
X_hidden_t2      = hidden_test_tier2[FEATURES_T2]
y_hidden_true_t2 = y_hidden_true.values  # sama persis dengan Approach 1

In [5]:
def objective_tier2(trial):
    params = {
        'n_estimators'     : 300,
        'learning_rate'    : trial.suggest_float('learning_rate', 0.01, 0.1, step=0.01),
        'num_leaves'       : trial.suggest_int('num_leaves', 31, 127),
        'max_depth'        : trial.suggest_int('max_depth', 6, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'random_state'     : RANDOM_SEED,
        'n_jobs'           : -1,
        'verbose'          : -1,
    }

    model = lgb.LGBMRegressor(**params)
    model.fit(X_train_t2, y_train_t2)

    # Kalibrasi per-shop dari anchor (copy eksplisit agar tidak mutate df asli)
    anchor_eval = anchor_tier2[['shopId', 'price']].copy()
    anchor_eval['pred'] = np.expm1(model.predict(X_anchor_t2))
    anchor_eval['ratio'] = anchor_eval['price'] / (anchor_eval['pred'] + 1e-5)

    # Shop-level calibration — fallback ke global kalau shop tidak ada di anchor
    shop_calib   = anchor_eval.groupby('shopId')['ratio'].median().to_dict()
    global_calib = float(np.median(anchor_eval['ratio']))

    # Prediksi hidden test
    preds_base  = np.expm1(model.predict(X_hidden_t2))
    # Di objective_tier2
    calib_map = hidden_test_tier2['shopId'].astype(str).map(
        {str(k): v for k, v in shop_calib.items()}
    ).fillna(global_calib)

    preds_calib = preds_base * calib_map.values

    return mean_absolute_percentage_error(y_hidden_true_t2, preds_calib)

print("--- Starting Optuna Tuning (Approach 2: Entity-Conditioned) ---")
study_t2 = optuna.create_study(direction='minimize')
study_t2.optimize(objective_tier2, n_trials=50)  # naikkan dari 20 → 50

print(f"\nBest MAPE (Approach 2, shop-calibrated): {study_t2.best_value*100:.2f}%")
print(f"Best params: {study_t2.best_params}")

--- Starting Optuna Tuning (Approach 2: Entity-Conditioned) ---

Best MAPE (Approach 2, shop-calibrated): 1.57%
Best params: {'learning_rate': 0.09, 'num_leaves': 50, 'max_depth': 11, 'min_child_samples': 32}


In [6]:
# =====================================================================
# EVALUASI FINAL — APPROACH 2
# =====================================================================

best_model_t2 = lgb.LGBMRegressor(
    n_estimators=300, random_state=RANDOM_SEED, n_jobs=-1, verbose=-1,
    **study_t2.best_params
)
best_model_t2.fit(X_train_t2, y_train_t2)

# Rebuild kalibrasi dari anchor
anchor_final = anchor_tier2[['shopId', 'price']].copy()
anchor_final['pred']  = np.expm1(best_model_t2.predict(X_anchor_t2))
anchor_final['ratio'] = anchor_final['price'] / (anchor_final['pred'] + 1e-5)

final_shop_calib  = anchor_final.groupby('shopId')['ratio'].median().to_dict()
final_global_calib = float(np.median(anchor_final['ratio']))

# Coverage report — berapa % hidden test ter-cover shop-level calibration
covered = hidden_test_tier2['shopId'].isin(final_shop_calib).mean() * 100
print(f"Shop-level calibration coverage: {covered:.1f}% of hidden test rows")

# Prediksi final
final_base_t2   = np.expm1(best_model_t2.predict(X_hidden_t2))
# Di evaluasi final — sama
calib_map_final = hidden_test_tier2['shopId'].astype(str).map(
        {str(k): v for k, v in final_shop_calib.items()}
    ).fillna(final_global_calib)
final_calib_t2  = final_base_t2 * calib_map_final.values

print("\n--- FINAL EVALUATION (Approach 2: Entity-Conditioned) ---")
print(f"{'Metric':<6}  {'Baseline':>14}  {'Shop-Calibrated':>16}")
print("-" * 42)

for name, func in [
    ('MAE',  mean_absolute_error),
    ('RMSE', lambda y, p: np.sqrt(mean_squared_error(y, p))),
    ('MAPE', lambda y, p: mean_absolute_percentage_error(y, p) * 100),
]:
    base  = func(y_hidden_true_t2, final_base_t2)
    calib = func(y_hidden_true_t2, final_calib_t2)
    unit  = '%' if name == 'MAPE' else ''
    print(f"{name:<6}  {base:>13.2f}{unit}  {calib:>15.2f}{unit}")

mape_base  = mean_absolute_percentage_error(y_hidden_true_t2, final_base_t2)
mape_calib = mean_absolute_percentage_error(y_hidden_true_t2, final_calib_t2)
improvement = (mape_base - mape_calib) / mape_base * 100
print(f"\nCalibration reduced MAPE by : {improvement:.2f}%")
print(f"Global fallback calib factor: {final_global_calib:.4f}")

# =====================================================================
# PERBANDINGAN APPROACH 1 vs APPROACH 2
# =====================================================================
print("\n--- APPROACH COMPARISON (Outage Simulation) ---")
print(f"{'':30} {'Approach 1':>12}  {'Approach 2':>12}")
print("-" * 58)

a1_mape = mean_absolute_percentage_error(y_hidden_true, final_preds_calib) * 100
a2_mape = mape_calib * 100
a1_mae  = mean_absolute_error(y_hidden_true, final_preds_calib)
a2_mae  = mean_absolute_error(y_hidden_true_t2, final_calib_t2)

print(f"{'MAPE (calibrated)':<30} {a1_mape:>11.2f}%  {a2_mape:>11.2f}%")
print(f"{'MAE  (calibrated)':<30} {a1_mae:>12,.0f}  {a2_mae:>12,.0f}")
print(f"{'Calibration granularity':<30} {'global':>12}  {'per-shop':>12}")
print(f"{'Shop coverage':<30} {'100%':>12}  {covered:>10.1f}%")

Shop-level calibration coverage: 87.2% of hidden test rows

--- FINAL EVALUATION (Approach 2: Entity-Conditioned) ---
Metric        Baseline   Shop-Calibrated
------------------------------------------
MAE         509917.79        504283.73
RMSE       2216848.75       2201854.90
MAPE             1.59%             1.57%

Calibration reduced MAPE by : 1.63%
Global fallback calib factor: 0.9993

--- APPROACH COMPARISON (Outage Simulation) ---
                                 Approach 1    Approach 2
----------------------------------------------------------
MAPE (calibrated)                     1.21%         1.57%
MAE  (calibrated)                   415,530       504,284
Calibration granularity              global      per-shop
Shop coverage                          100%        87.2%


## Validation Summary

### Setup
- **Training data:** 301,355 rows (before 2025-03-22)
- **Validation day:** 2025-03-22 (4,871 rows — outage simulation)
- **Simulation design:** 100 organic anchor rows (with price) + 4,771 hidden test rows (stripped to 4 columns, backfilled via LOCF from train)
- **Cold-start cells patched:** 12,315 (median/mode fallback)

---

### Results

| Metric | A1 Baseline | A1 Calibrated | A2 Baseline | A2 Shop-Calibrated |
|--------|---:|---:|---:|---:|
| MAE    | 415,674.82 | 415,530.29 | 509,917.79 | 504,283.73 |
| RMSE   | 2,338,620.36 | 2,338,473.11 | 2,216,848.75 | 2,201,854.90 |
| MAPE   | 1.21% | **1.21%** | 1.59% | **1.57%** |

- **Best hyperparams A1:** `lr=0.08, num_leaves=80, max_depth=11, min_child_samples=58`
- **Best hyperparams A2:** `lr=0.09, num_leaves=50, max_depth=11, min_child_samples=32`
- **Shop-level calibration coverage (A2):** 87.2% — 12.8% rows fallback ke global factor

---

### Approach Comparison

|  | Approach 1 | Approach 2 |
|--|---:|---:|
| MAPE (calibrated) | **1.21%** | 1.57% |
| MAE (calibrated) | **415,530** | 504,284 |
| Calibration granularity | global | per-shop |
| Shop coverage | 100% | 87.2% |

---

### Findings

**Approach 1 (Global Model) unggul** di semua metrik utama — MAPE 1.21% vs 1.57%, MAE 415k vs 504k IDR.

Ada dua insight menarik dari hasil ini:

**1. Calibration factor A1 praktis = 1.0000**

Artinya model global sudah sangat well-calibrated secara baseline — tidak ada systematic over/under-prediction yang perlu dikoreksi. Ini menjelaskan mengapa calibration hanya mengubah MAPE sebesar -0.02% (noise level). Model sudah "tahu" skala harga marketplace dengan baik tanpa perlu koreksi eksternal.

**2. Shop-level calibration A2 hanya reduce MAPE 1.63%**

Dengan hanya 100 anchor, banyak toko tidak terwakili — 12.8% hidden test rows terpaksa fallback ke global factor (0.9993, hampir 1.0). Shop-level calibration ratio menjadi noisy karena dihitung dari 1–2 data points per toko. Historical stats sebagai prior (`historical_shop_price_mean`) menambah noise pada toko dengan sedikit historis, bukan sinyal yang bersih.

Menariknya, A2 menghasilkan **RMSE lebih rendah** dari A1 (2.20M vs 2.34M), menunjukkan entity stats membantu meredam outlier harga tinggi — namun tidak cukup untuk mengkompensasi noise di MAPE dan MAE.

---

### Decision

**Approach 1 (Global Marketplace Model) dipilih untuk inference ke test set.**

Approach 2 tetap relevan secara konseptual untuk skenario dengan anchor lebih besar (misalnya 500+ rows per hari) sehingga shop-level calibration bisa diestimasi dengan reliable. Dalam kondisi outage dengan hanya 100 anchor, granularitas per-toko menjadi liabilitas dibanding keunggulan.